# 🐭 Canlı Modellerden Koda · Uygulama II
### Sedef (psoriasis) modelinde fotoğraftan şiddet skoruna

**TÜBİTAK-BİDEB 2237 · Uygulamalı Eğitimle Sağlık Araştırmalarında Yapay Zekâ ve Hukuki Çerçevesi**
KTÜ İLAFAR · 24 Ekim 2026 · Dr. Öğr. Üyesi Ramazan Özgür Doğan

---

**Nasıl kullanılır?** Gri kutulardaki **▶** düğmesine bas (ya da **Shift + Enter**). Hücreleri **sırayla** çalıştır.

> 🩺 **Sağlıkçılar için:** Amaç kodu ezberlemek değil; *deneyinizdeki fotoğrafların nasıl veriye dönüştüğünü* görmek.
> 💻 **Bilgisayarcılar için:** Gizli hücreleri açıp tüm kodu inceleyebilirsin.

**Veri:** PCL-PASI'nin eğitim için hazırlanmış **küçük demo alt kümesi** (565 görüntü + 24.766 görüntünün etiket tablosu).
Tam veri seti: https://huggingface.co/datasets/pharmaiktu/pcl-pasi · Lisans: **CC BY-NC 4.0** · Etik onay: 2024/33, KTÜ HADYEK.
⚠️ Yalnız bu eğitim için paylaşılmıştır; klinik kullanım için değildir.

## 0 · Hazırlık

In [ ]:
#@title ▶ 0 · Kütüphaneleri yükle { display-mode: "form" }
import os, glob, zipfile, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})
SKOR = {0: "Yok", 1: "Hafif", 2: "Şiddetli"}
GRUP_TR = {"HC-N": "Sağlıklı kontrol", "HC-TCH": "Sağlıklı + test bileşiği (yüksek)", "PM-U": "Model (tedavisiz)",
           "PM-TCL": "Model + test (düşük doz)", "PM-TCM": "Model + test (orta doz)", "PM-TCH": "Model + test (yüksek doz)",
           "PM-SR": "Model + standart referans"}
print("Hazır ✓")

## 1 · Veriyi getir
Eğitmenin paylaştığı **Google Drive bağlantısını** (dosya ya da klasör) aşağıdaki kutuya yapıştır ve hücreyi çalıştır.
Bağlantı yoksa kutuyu boş bırak: hücre senden **zip dosyasını yüklemeni** isteyecek.

In [ ]:
#@title ▶ 1 · Demo veriyi indir / yükle { display-mode: "form" }
VERI_LINKI = ""  #@param {type:"string"}
if not os.path.exists("pcl_pasi_demo/demo_metadata.csv"):
    link = VERI_LINKI.strip()
    if link:
        import gdown, requests
        if "drive.google.com" not in link:   # kısa bağlantı (ör. tinyurl): gerçek Drive adresine çöz
            link = requests.get(link, allow_redirects=True, timeout=30).url
        if "/folders/" in link:              # klasör bağlantısı: içinde zip varsa yalnız onu indir
            try:
                zipler = [f for f in gdown.download_folder(url=link, skip_download=True, quiet=True) if f.path.endswith(".zip")]
            except Exception:
                zipler = []
            if zipler:
                gdown.download(id=zipler[0].id, output="pcl_pasi_demo.zip", quiet=False)
            else:                            # zip yoksa klasörü olduğu gibi indir (birkaç dakika sürebilir)
                gdown.download_folder(url=link, output="pcl_pasi_demo", quiet=True)
        else:
            gdown.download(url=link, output="pcl_pasi_demo.zip", quiet=False, fuzzy=True)
    elif not os.path.exists("pcl_pasi_demo.zip"):
        from google.colab import files
        files.upload()                       # pcl_pasi_demo.zip dosyasını seç
    if not os.path.exists("pcl_pasi_demo/demo_metadata.csv"):
        zipfile.ZipFile("pcl_pasi_demo.zip").extractall(".")
demo = pd.read_csv("pcl_pasi_demo/demo_metadata.csv")
tum = pd.read_csv("pcl_pasi_demo/full_labels.csv")
print(f"Demo: {len(demo)} görüntü · Tüm etiket tablosu: {len(tum):,} görüntü ✓".replace(",", "."))

## 2 · Önce tabloya bakalım (görüntüsüz!)
Bir veri setinin kalbi **metadata**'dır: her fotoğraf için *hangi fare, hangi grup, hangi gün, hangi kamera, hangi skor*.

In [ ]:
tum.head(8)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
tum.camera_id.value_counts().sort_index().plot.bar(ax=ax[0], color="#00426a"); ax[0].set_title("Kamera başına görüntü")
tum.groupby("group_id").subject_id.nunique().plot.bar(ax=ax[1], color="#cd061e"); ax[1].set_title("Grup başına fare")
tum.groupby("day").size().plot.bar(ax=ax[2], color="#0ea5e9"); ax[2].set_title("Gün başına görüntü")
for a in ax: a.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
print("Fare sayısı:", tum.subject_id.nunique(), "· gün:", tum.day.nunique(), "· resmi bölme:", tum.drop_duplicates("subject_id").split.value_counts().to_dict())

### Skorlar zamanla nasıl değişiyor?
Üç skor var: **E**ritem (kızarıklık), **T**hickness (kalınlık), **S**caling (pullanma). Her biri 0 = yok, 1 = hafif, 2 = şiddetli.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for a, skor in zip(ax, ["erythema", "thickness", "scaling"]):
    tablo = tum.pivot_table(index="day", columns="group_id", values=skor, aggfunc="mean")
    for gid in ["HC-N", "PM-U", "PM-TCL", "PM-TCM", "PM-TCH", "PM-SR", "HC-TCH"]:
        a.plot(tablo.index, tablo[gid], marker="o", lw=2.5 if gid in ("HC-N", "PM-U") else 1.4, label=GRUP_TR[gid])
    a.set_title(skor.capitalize()); a.set_xlabel("deney günü"); a.set_xticks(range(1, 12))
ax[0].set_ylabel("ortalama skor (0–2)"); ax[2].legend(fontsize=7.5, loc="upper left")
plt.tight_layout(); plt.show()

🩺 **Okuma:** Sağlıklı kontrol sıfıra yakın kalırken model grubu günler içinde yükseliyor. Bu grafiği **tek satır görüntü işlemeden**,
yalnız iyi kurulmuş bir etiket tablosundan çizdik. İyi tasarlanmış deney + düzenli kayıt = yarı yarıya analiz.

## 3 · Görüntülerle tanışma

In [ ]:
#@title ▶ Yardımcı: görüntü göster { display-mode: "form" }
def goster(satirlar, baslik, n_sutun=6, boy=2.6):
    satirlar = list(satirlar.itertuples()) if isinstance(satirlar, pd.DataFrame) else satirlar
    n = len(satirlar); n_satir = int(np.ceil(n / n_sutun))
    fig, ax = plt.subplots(n_satir, n_sutun, figsize=(n_sutun * boy, n_satir * boy * 0.78))
    ax = np.atleast_1d(ax).ravel()
    for a in ax: a.axis("off")
    for a, r in zip(ax, satirlar):
        a.imshow(Image.open("pcl_pasi_demo/" + r.file))
        a.set_title(f"{r.subject_id} · gün {r.day}\nE{r.erythema} T{r.thickness} S{r.scaling}", fontsize=8)
    fig.suptitle(baslik, fontweight="bold"); plt.tight_layout(); plt.show()

In [ ]:
# Aynı fare, 11 günlük deney boyunca (model grubu, tedavisiz)
fare = demo[(demo.group_id == "PM-U") & ~demo.kamera_uclusu].groupby("subject_id").day.nunique().idxmax()   # en çok günü olan fare
zaman = demo[(demo.subject_id == fare) & ~demo.kamera_uclusu].sort_values(["day", "image_id"]).drop_duplicates("day")
goster(zaman, f"{fare} · {GRUP_TR['PM-U']} · zaman içinde", n_sutun=len(zaman), boy=3.2)

In [ ]:
# Aynı gün: sağlıklı kontrol ile model grubunu yan yana koyalım (7. gün)
g7 = demo[(demo.day == 7) & ~demo.kamera_uclusu]
yan = pd.concat([g7[g7.group_id == "HC-N"].drop_duplicates("subject_id").head(3), g7[g7.group_id == "PM-U"].drop_duplicates("subject_id").head(3)])
goster(yan, "7. gün · üst sıra: sağlıklı kontrol · sağ: model")

In [ ]:
# Aynı fare, aynı gün, ÜÇ FARKLI KAMERA
uc = demo[demo.kamera_uclusu].sort_values(["subject_id", "camera_id"])
satir = [r for r in uc.itertuples()]
fig, ax = plt.subplots(1, len(satir), figsize=(2.6 * len(satir), 2.4))
for a, r in zip(np.atleast_1d(ax), satir):
    a.imshow(Image.open("pcl_pasi_demo/" + r.file)); a.set_title(f"{r.subject_id} · {r.camera_id}", fontsize=8); a.axis("off")
plt.suptitle("CAM1 = Sony α6500 · CAM2 = Xiaomi 12 Lite · CAM3 = iPhone 15 Pro", fontweight="bold"); plt.tight_layout(); plt.show()

📸 Aynı deri, üç cihazda üç farklı renk ve ışık. Bir model bu farka takılırsa **"hastalığı" değil "kamerayı"** öğrenir. Bunu birazdan göreceğiz.

## 4 · Kızarıklığı bir sayıyla ölçmek
En basit fikir: Görüntünün ortasındaki derinin **ne kadar kırmızı** olduğunu ölçelim (kırmızı kanal − yeşil kanal).
Eritem skoru arttıkça bu sayı da artıyor mu?

In [ ]:
def kirmizilik(yol):
    a = np.asarray(Image.open(yol).convert("RGB")).astype(float)
    h, w = a.shape[:2]; m = a[h // 4: 3 * h // 4, w // 4: 3 * w // 4]       # merkezdeki yarı alan
    return (m[..., 0] - m[..., 1]).mean()
demo["kirmizilik"] = [kirmizilik("pcl_pasi_demo/" + f) for f in demo.file]
fig, ax = plt.subplots(figsize=(6.5, 4))
demo.boxplot(column="kirmizilik", by="erythema", grid=False, ax=ax)
ax.set_title("Kırmızılık indeksi · eritem skoruna göre"); fig.suptitle(""); ax.set_xlabel("eritem skoru (0 yok · 1 hafif · 2 şiddetli)"); ax.set_ylabel("R − G")
plt.show()
print("Spearman korelasyonu:", round(demo[["kirmizilik", "erythema"]].corr(method="spearman").iloc[0, 1], 3))

✅ Basit bir renk ölçüsü bile skorla ilişkili. ❌ Ama kutular üst üste biniyor: ışık, kamera, kürk, açı… Tek bir sayı yetmez.
**Derin öğrenme**, binlerce görsel özelliği kendisi çıkarır.

## 5 · Derin öğrenme ile özellik çıkarmak (transfer öğrenme)
ImageNet'te 1,2 milyon fotoğrafla eğitilmiş hazır bir ağ (**ResNet-18**) kullanıyoruz. Ağın son katmanını atıp her fotoğrafı
**512 sayılık bir parmak izine** dönüştürüyoruz. Eğitim yok, yalnız "bakma" — CPU'da birkaç dakika sürebilir.

In [ ]:
import torch, torchvision
from torchvision.models import resnet18, ResNet18_Weights
agirlik = ResNet18_Weights.DEFAULT
ag = resnet18(weights=agirlik); ag.fc = torch.nn.Identity(); ag.eval()
donustur = agirlik.transforms()
oz = []
with torch.no_grad():
    for i in range(0, len(demo), 32):
        parti = torch.stack([donustur(Image.open("pcl_pasi_demo/" + f).convert("RGB")) for f in demo.file[i:i + 32]])
        oz.append(ag(parti).numpy())
X = np.concatenate(oz); print("Özellik matrisi:", X.shape, "→ her görüntü 512 sayı")

## 6 · En önemli ders: **veriyi doğru bölmek**
Her çekim oturumunda bir fareden art arda birçok kare çekiliyor; ardışık iki kare **neredeyse aynı**. Şimdi çok basit bir "model"
kuralım: *"Test fotoğrafına en çok benzeyen eğitim fotoğrafını bul, onun skorunu söyle."* (1-en yakın komşu). Bunu iki farklı sınavla ölçelim:

* ❌ **Rastgele bölme:** Fotoğraflar rastgele eğitim/test diye ayrılır → aynı oturumun ikiz karesi büyük ihtimalle eğitimde.
* ✅ **Fare düzeyinde bölme:** Test farelerini model **hiç görmez**. Gerçek hayattaki durum budur: yeni hayvan, yeni hasta.

Sonucun şansa bağlı olmaması için her sınavı 10 kez farklı bölmelerle tekrarlıyoruz.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

y = demo.erythema.values
def en_benzer(): return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))
rastgele, fare_duzeyi = [], []
for tohum in range(10):
    a, b = train_test_split(np.arange(len(demo)), test_size=0.3, random_state=tohum, stratify=y)
    rastgele.append(accuracy_score(y[b], en_benzer().fit(X[a], y[a]).predict(X[b])))
    a, b = next(GroupShuffleSplit(test_size=0.3, random_state=tohum).split(X, groups=demo.subject_id))
    fare_duzeyi.append(accuracy_score(y[b], en_benzer().fit(X[a], y[a]).predict(X[b])))
print(f"❌ Rastgele bölme    : ortalama doğruluk %{100 * np.mean(rastgele):.1f}")
print(f"✅ Fare düzeyinde    : ortalama doğruluk %{100 * np.mean(fare_duzeyi):.1f}")
fig, ax = plt.subplots(figsize=(5.6, 3.8))
ax.boxplot([np.array(rastgele) * 100, np.array(fare_duzeyi) * 100], widths=0.5)
ax.set_xticks([1, 2], ["Rastgele\n(sızıntılı)", "Fare düzeyinde\n(dürüst)"]); ax.set_ylabel("doğruluk (%)")
ax.set_title("Aynı model, iki farklı sınav (10'ar tekrar)"); plt.show()

🤔 **Fark nereden geliyor?** Model aslında *fareyi tanıyor*. Aynı yöntemle bu kez skoru değil, **fotoğraftaki farenin kimliğini**
tahmin ettirelim. 43 fare var; rastgele tahminle tutturma şansı yalnız **%2,3**.

In [ ]:
kimlik = demo.subject_id.values
a, b = train_test_split(np.arange(len(demo)), test_size=0.3, random_state=0)
kimlik_dogru = accuracy_score(kimlik[b], en_benzer().fit(X[a], kimlik[a]).predict(X[b]))
print(f"Fare kimliği doğruluğu: %{100 * kimlik_dogru:.1f}   (şans: %{100 / demo.subject_id.nunique():.1f})")

🎓 **Ders:** Model hastalığı değil **bireyi** ezberleyebilir. Rastgele bölme bu ezberi ödüllendirir ve gerçek hayatta karşılaşılmayacak
kadar iyimser sonuç verir. Bu yüzden PCL-PASI'nin resmi bölmesi **fare düzeyindedir** ve uyarısı açıktır:
*"Kameraya göre bölme; birey düzeyinde böl."* Aynı kural hastalar için de geçerli: **aynı hastanın görüntüleri hem eğitimde hem testte olmamalı.**

### Dürüst model
Şimdi resmi bölmeyle (29 eğitim faresi ↔ 14 hiç görülmemiş fare) bir **lojistik regresyon** modeli kuralım.

In [ ]:
def model_kur(): return make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.05))
egitim = (demo.split == "train").values
m2 = model_kur().fit(X[egitim], y[egitim]); tahmin = m2.predict(X[~egitim])
print(f"Eritem · hiç görülmemiş {demo[~egitim].subject_id.nunique()} fare · doğruluk %{100 * accuracy_score(y[~egitim], tahmin):.1f}")
fig, ax = plt.subplots(figsize=(4.6, 4))
ConfusionMatrixDisplay.from_predictions(y[~egitim], tahmin, display_labels=[SKOR[k] for k in sorted(SKOR)], cmap="Blues", colorbar=False, ax=ax)
ax.set_title("Eritem · karışıklık matrisi"); plt.show()

## 7 · Üç skor birden (çok görevli bakış)
Aynı 512 sayılık parmak izinden kalınlık ve pullanmayı da tahmin edelim.

In [ ]:
satir = []
for skor in ["erythema", "thickness", "scaling"]:
    yy = demo[skor].values
    m = model_kur().fit(X[egitim], yy[egitim]); p = m.predict(X[~egitim])
    satir.append([skor, accuracy_score(yy[~egitim], p), f1_score(yy[~egitim], p, average="macro")])
pd.DataFrame(satir, columns=["skor", "doğruluk", "makro F1"]).round(3)

## 8 · Modelin gözünden
Test farelerinden birkaç fotoğraf: üstte **gerçek**, altta **modelin tahmini**.

In [ ]:
test = demo[~egitim].copy(); test["tahmin"] = tahmin
ornek = test.sample(8, random_state=3)
fig, ax = plt.subplots(2, 4, figsize=(13, 6.2))
for a, r in zip(ax.ravel(), ornek.itertuples()):
    a.imshow(Image.open("pcl_pasi_demo/" + r.file)); a.axis("off")
    renk = "#16a34a" if r.erythema == r.tahmin else "#dc2626"
    a.set_title(f"gerçek: {SKOR[r.erythema]}\nmodel: {SKOR[r.tahmin]}", color=renk, fontsize=9)
plt.suptitle("Eritem tahmini · yeşil = doğru, kırmızı = yanlış", fontweight="bold"); plt.tight_layout(); plt.show()

## 🎯 Toparlarsak
| Adım | Ne yaptık? | Deneyinize karşılığı |
|:--|:--|:--|
| Metadata | Grup × gün × skor eğrileri | Deney tasarımı ve düzenli kayıt |
| Kamera üçlüsü | Cihaz farkını gördük | Standart çekim protokolü |
| Kırmızılık indeksi | Tek sayılık ölçüm | Hızlı, ama sınırlı |
| ResNet-18 özellikleri | Transfer öğrenme | Az veriyle güçlü başlangıç |
| Doğru bölme | Sızıntıyı yakaladık | Yeni hayvana / hastaya genelleme |

**Tam ölçekte:** 24.766 görüntünün tamamı, GPU'da uçtan uca eğitilen çok görevli bir ağ (E/T/S birlikte) ve açıklanabilirlik
haritaları (model derinin neresine bakıyor?).

*Kaynak: PharmAI KTU Research Group (2026). PCL-PASI: Multi-Task Multi-Device Psoriasis Dataset. https://huggingface.co/datasets/pharmaiktu/pcl-pasi (CC BY-NC 4.0).*